In [ ]:
import copy
import pandas as pd
from typing import Dict, List, Tuple, Optional, Any
import numpy as np
import torch
import torch.nn as nn
from torch.nn.utils import clip_grad_norm_
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel, BertForSequenceClassification
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
import random

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Устройство: {device}")

# Установление seed для воспроизводимости
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

Устройство: cuda


#1.Подготовка данных

Загружаем тренировочный и тестовый датасеты.  


In [ ]:
train_df = pd.read_parquet('train.parquet')
test_df = pd.read_parquet('test.parquet')
train_df

,text,integrity,integrity_reasoning,factuality,factuality_reasoning,truthfulness,truthfulness_reasoning
uuid,,,,,,,
dff0d182-5434-46e2-9183-be278f66667f,"Соцветия ромашки, которые продаются в аптеках,...",1.0,The content is an informative text about the u...,1.0,The content provides informative and fact-base...,1.0,The content provides credible information abou...
8268f315-03db-4f12-aa46-0b968c3b1b19,Кто из черниговцев сам будет убирать придомову...,1.0,The content is an informative text about a dec...,1.0,The content is a coherent and informative news...,1.0,The content provides a detailed and credible a...
dc7cd0dd-9eca-418e-8356-9050c4a17cdc,Тамбовчане смогут услышать романсы 20-30 годов...,1.0,This is an informative announcement about a cu...,1.0,The content provides detailed information abou...,1.0,The content provides credible information abou...
e9f43939-22b1-4a4e-ab5a-34a25b269039,Человек может отказаться от ТВ и проигрывателе...,1.0,The content is an informative text discussing ...,1.0,The content is a detailed exploration of super...,0.0,The content is based on superstitions and esot...
412b31bb-ba09-44ae-abd7-49c9b783b005,Простая интеграция в системы PROFINET®\nВозмож...,1.0,The content is an informative text about a PRO...,1.0,The content provides a detailed and coherent d...,1.0,The content provides a detailed and coherent d...
...,...,...,...,...,...,...,...
13b93d68-1dd4-49c9-b1d9-310e42633370,Ключевые результаты за 1 квартала 2020 года:\n...,1.0,The content is an informative text providing k...,1.0,The content provides a detailed and fact-based...,0.5,The content is a financial report with figures...
82f28832-78f7-4b81-bd89-af515d072c5d,Диван SleepArt Алтгерия по цене 66 000 руб. с ...,1.0,This is a description of a single item from a ...,1.0,The content provides a detailed description of...,0.5,The content is a brief product description and...
254cb352-1fa4-4847-9f42-b85bbadc6250,Так что ТС в таких странах будет бесполезна.\n...,1.0,The content is an informative text discussing ...,1.0,The content is a coherent discussion and analy...,0.5,The content is a subjective discussion and ana...


Фильтруем объекты, у которых хотя бы одна метка равна 0.5 (неопределённые примеры).  
Создаём стратифицированную разметку для корректного разбиения на train/val с сохранением пропорций классов.


In [ ]:
def filter_object(df: pd.DataFrame) -> pd.DataFrame:
    """
    Фильтрует строки, где хотя бы одна метка равна 0.5.
    Выводит статистику до и после фильтрации.
    """
    df_filtered = df.copy()
    mask = (
        (df_filtered['integrity'] == 0.5) |
        (df_filtered['factuality'] == 0.5) |
        (df_filtered['truthfulness'] == 0.5)
        )
    df_filtered = df_filtered[~mask]

    print(f"Всего объектов: {len(df)}")
    print(f"Объектов после фильтрации 0.5: {len(df_filtered)}")
    print(f"Удалено: {len(df) - len(df_filtered)}")
    print("Распределение классов после фильтрации:")
    for column in ['integrity', 'factuality', 'truthfulness']:
        counts = df_filtered[column].value_counts().sort_index()
        zeros = counts.get(0, 0)
        ones = counts.get(1, 0)
        balance = ones / len(df_filtered) * 100
        print(f"  {column}: 0={zeros}, 1={ones} ({balance:.1f}%)")

    return df_filtered

# Очищение тестового датафрейма от объектов с меткой 0.5
train_df_filtered = filter_object(train_df)

# Создание комбинированной метки для последующей стратификации
train_df_filtered['stratify_label'] = (
    train_df_filtered['integrity'].astype(str) +
    train_df_filtered['factuality'].astype(str) +
    train_df_filtered['truthfulness'].astype(str)
)

# Стратифицированное разбиение данных на обучающую и валидационную части
train_data, val_data = train_test_split(
    train_df_filtered,
    test_size=0.15,
    random_state=42,
    stratify=train_df_filtered['stratify_label']
)

# Удаление временной колонки комбинированной метки
train_data = train_data.drop(columns=['stratify_label'])
val_data = val_data.drop(columns=['stratify_label'])

print(f"\nРазделение данных:")
print(f"Train: {len(train_data)} примеров")
print(f"Validation: {len(val_data)} примеров")

Всего объектов: 15000
Объектов после фильтрации 0.5: 9352
Удалено: 5648
Распределение классов после фильтрации:
  integrity: 0=1196, 1=8156 (87.2%)
  factuality: 0=1093, 1=8259 (88.3%)
  truthfulness: 0=851, 1=8501 (90.9%)

Разделение данных:
Train: 7949 примеров
Validation: 1403 примеров


Создаём кастомный датасет для RuBERT:  
- Токенизируем текст (с добавлением reasoning только для train)  
- Возвращаем input_ids, attention_mask и отдельные метки (long) для каждой задачи  
- Настраиваем DataLoader'ы

In [ ]:
class CustomDataset(Dataset):
    """Кастомный датасет для multi-task классификации на основе RuBERT."""
    def __init__(self, df, tokenizer, max_length=512, use_reasoning=False) -> None:
        self.df = df.reset_index(drop=True) # копирование данных с переиндексацией (сбросили старые индексы для перестраховки)
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.use_reasoning = use_reasoning

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx)-> Dict[str, torch.Tensor]:
        """
        Возвращает один пример датасета. А именно словарь с:
                - input_ids, attention_mask — токенизированный текст
                - integrity, factuality, truthfulness — метки
        """
        text = str(self.df.iloc[idx]['text']).strip() # текст объекта
        if self.use_reasoning:
            reasoning_parts = []
            for task in ['integrity', 'factuality', 'truthfulness']:
                col = f"{task}_reasoning"
                if col in self.df.columns:
                    reasoning = str(self.df.iloc[idx][col])
                    if reasoning and reasoning.lower() != 'nan':
                        reasoning_parts.append(f"[{task} reasoning]: {reasoning}")
            if reasoning_parts:
                text += "  " + "  ".join(reasoning_parts) # добавление reasoning к тексту объекта

        # Получение меток
        labels = {
            'integrity': int(self.df.iloc[idx]['integrity']),
            'factuality': int(self.df.iloc[idx]['factuality']),
            'truthfulness': int(self.df.iloc[idx]['truthfulness'])
        }

        # Токенизация текста
        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'integrity': torch.tensor(labels['integrity'], dtype=torch.long),
            'factuality': torch.tensor(labels['factuality'], dtype=torch.long),
            'truthfulness': torch.tensor(labels['truthfulness'], dtype=torch.long)
        }


# Инициализация токенизатора
MODEL_NAME = 'DeepPavlov/rubert-base-cased'
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

# Создание датасетов
train_dataset = CustomDataset(
    train_data,
    tokenizer,
    max_length=512,
    use_reasoning=True
)

val_dataset = CustomDataset(
    val_data,
    tokenizer,
    max_length=512,
    use_reasoning=False # на валидации не используем reasoning, чтобы не завышать метрики
)

# DataLoader'ы
BATCH_SIZE = 8
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

#2.Определение модели

Создаём много-задачную модель (с тремя классификационными головами) на базе ruBERT:  
- Берём предобученный `DeepPavlov/rubert-base-cased`  
- Добавляем общий shared-слой для извлечения общих признаков (полезных для всех трех задач)
- Три независимые головы классификации (по одной на каждую задачу)  
- Выводим количество обучаемых параметров для контроля
Python

In [ ]:
class ThreeHeadRuBERT(nn.Module):
    """
    Модель на базе ruBERT с тремя независимыми бинарными головами для задач integrity, factuality, truthfulness
    """
    def __init__(self, model_name='DeepPavlov/rubert-base-cased', dropout_rate=0.2) -> None:
        super().__init__()
        self.bert = BertModel.from_pretrained(model_name)
        hidden_size = self.bert.config.hidden_size
        self.dropout = nn.Dropout(0.15)

        # Извлекаем общие признаки
        self.shared_layer = nn.Linear(hidden_size, 256)
        self.shared_activation = nn.GELU()
        self.shared_dropout = nn.Dropout(dropout_rate)

        # Три независимые классификационные головы
        self.integrity_classifier = nn.Sequential(
            nn.Linear(256, 128),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(128, 2)
        )

        self.factuality_classifier = nn.Sequential(
            nn.Linear(256, 128),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(128, 2)
        )

        self.truthfulness_classifier = nn.Sequential(
            nn.Linear(256, 128),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(128, 2)
        )

    def forward(self, input_ids, attention_mask) -> Dict[str, torch.Tensor]:
        """
        Прямой проход модели. В аргументах:
            input_ids: тензор токенов [batch_size, seq_len]
            attention_mask: маска внимания [batch_size, seq_len]
        Возвращает: словарь с логитами для каждой задачи
        """
        # Получаем эмбеддинги от BERT
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True
        )

        # Используем [CLS] токен
        pooled_output = outputs.pooler_output
        pooled_output = self.dropout(pooled_output)

        # Извлекаем общие признаки через shared слой
        shared_features = self.shared_layer(pooled_output)
        shared_features = self.shared_activation(shared_features)
        shared_features = self.shared_dropout(shared_features)

        # Прогоняем через три независимые классификационные головы
        return {
            'integrity': self.integrity_classifier(shared_features),
            'factuality': self.factuality_classifier(shared_features),
            'truthfulness': self.truthfulness_classifier(shared_features),
        }

# Инициализация модели
model = ThreeHeadRuBERT(MODEL_NAME, dropout_rate=0.2).to(device)
print(f"Модель загружена на устройство: {device}")
print(f"Количество обучаемых параметров: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")

config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: DeepPavlov/rubert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Модель загружена на устройство: cuda
Количество обучаемых параметров: 178149766


#3.Обучение и подсчёт метрик

In [ ]:
class Trainer:
    """Класс для обучения и оценки модели"""

    def __init__(self, model, device, tasks=['integrity', 'factuality', 'truthfulness'],
                 train_data=None) -> None:
        """
        Инициализирует объект Trainer для многозадачного обучения.
        """
        self.model = model
        self.device = device
        self.tasks = tasks

        # Вычисляем веса классов для каждой задачи
        self.loss_functions = {}
        if train_data is not None:
            from sklearn.utils.class_weight import compute_class_weight
            import numpy as np

            for task in tasks:
                labels = train_data[task].values
                weights = compute_class_weight('balanced', classes=np.array([0, 1]), y=labels)
                weights = torch.tensor(weights, dtype=torch.float).to(device)
                self.loss_functions[task] = nn.CrossEntropyLoss(weight=weights)
        else:
            self.loss_functions = {task: nn.CrossEntropyLoss() for task in tasks}

        self.learning_history = {
            'train_loss': [], 'val_loss': [],
            'val_f1':   {task: [] for task in tasks}
        }

    def train_epoch(self, dataloader, optimizer, scheduler=None, epoch_num=None, total_epochs=None) -> float:
        """
        Функция для обучения модели в течение эпохи
        """
        self.model.train()
        total_loss = 0
        pbar = tqdm(dataloader, desc=f'Epoch {epoch_num}/{total_epochs}')

        for batch in pbar:
            input_ids = batch['input_ids'].to(self.device)
            attention_mask = batch['attention_mask'].to(self.device)
            labels = {task: batch[task].to(self.device) for task in self.tasks}
            outputs = self.model(input_ids, attention_mask)
            losses = [self.loss_functions[task](outputs[task], labels[task]) for task in self.tasks]
            batch_loss = sum(losses) / len(losses)

            optimizer.zero_grad()
            batch_loss.backward()
            clip_grad_norm_(self.model.parameters(), 1.0)
            optimizer.step()
            if scheduler:
                scheduler.step()
            total_loss += batch_loss.item()
            pbar.set_postfix(loss=f'{total_loss / (pbar.n + 1):.4f}')

        avg_loss = total_loss / len(dataloader)
        return avg_loss

    def evaluate(self, dataloader) -> Tuple[float, Dict[str, float]]:
        """
        Функция выполняет оценку модели на валидационной выборке
        """
        self.model.eval()
        total_loss = 0
        all_preds  = {task: [] for task in self.tasks}
        all_labels = {task: [] for task in self.tasks}

        with torch.no_grad():
            pbar = tqdm(dataloader, desc='Evaluating', leave=False)

            for batch in pbar:
                input_ids = batch['input_ids'].to(self.device)
                attention_mask = batch['attention_mask'].to(self.device)
                labels = {task: batch[task].to(self.device) for task in self.tasks}
                outputs = self.model(input_ids, attention_mask)
                losses = [self.loss_functions[task](outputs[task], labels[task]) for task in self.tasks]
                batch_loss = sum(losses) / len(losses)
                total_loss += batch_loss.item()

                for task in self.tasks:
                    pred = torch.argmax(outputs[task], dim=1).cpu().numpy()
                    all_preds[task].extend(pred)
                    all_labels[task].extend(labels[task].cpu().numpy())

        avg_loss = total_loss / len(dataloader)
        f1_scores = {
            task: f1_score(all_labels[task], all_preds[task], average='binary', pos_label=1)
            for task in self.tasks
        }

        return avg_loss, f1_scores

    def train(self, train_loader, val_loader, epochs=10, lr=2e-5, patience=3):
        """
        Запускает полный цикл обучения модели
        """
        optimizer = AdamW(self.model.parameters(), lr=lr)
        best_val_f1 = -1
        patience_counter = 0
        best_state = None

        for epoch in range(1, epochs + 1):
            train_loss = self.train_epoch(
                train_loader, optimizer,
                epoch_num=epoch, total_epochs=epochs
            )

            val_loss, val_f1 = self.evaluate(val_loader)
            avg_val_f1 = np.mean(list(val_f1.values()))
            print(f"[{epoch:2d}/{epochs}]  train_loss: {train_loss:.4f}  val_loss: {val_loss:.4f}  val_f1: {avg_val_f1:.4f}")

            self.learning_history['train_loss'].append(train_loss)
            self.learning_history['val_loss'].append(val_loss)
            for task in self.tasks:
                self.learning_history['val_f1'][task].append(val_f1[task])

            if avg_val_f1 > best_val_f1:
                best_val_f1 = avg_val_f1
                best_state = copy.deepcopy(self.model.state_dict())
                patience_counter = 0
                torch.save(best_state, 'best_model.pth')
                print(f" → new best ({avg_val_f1:.4f})")
            else:
                patience_counter += 1
                print(f"  patience {patience_counter}/{patience}")
                if patience_counter >= patience:
                    print(f"Ранняя остановка на эпохе {epoch}")
                    break

        if best_state is not None:
            self.model.load_state_dict(best_state)
            print(f"Загружена лучшая модель  (val_f1 = {best_val_f1:.4f})")

        return self.learning_history

In [ ]:
# Создается экземпляр трейнера с передачей train_data для вычисления весов классов
trainer = Trainer(model, device, train_data=train_data)

# Обучение модели
metrics_history = trainer.train(
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=3,
    lr=3e-5,
    patience=1
)


Epoch 1/3:   0%|          | 0/994 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/176 [00:00<?, ?it/s]

[ 1/3]  train_loss: 0.5083  val_loss: 0.5751  val_f1: 0.9153
 → new best (0.9153)


Epoch 2/3:   0%|          | 0/994 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/176 [00:00<?, ?it/s]

[ 2/3]  train_loss: 0.3325  val_loss: 0.7925  val_f1: 0.9493
 → new best (0.9493)


Epoch 3/3:   0%|          | 0/994 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/176 [00:00<?, ?it/s]

[ 3/3]  train_loss: 0.3864  val_loss: 0.7842  val_f1: 0.9491
  patience 1/1
Ранняя остановка на эпохе 3
Загружена лучшая модель  (val_f1 = 0.9493)


#Прогон на тестовом датасете (создание submission)

In [ ]:
class TestDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=512):
        self.df = df
        self.tokenizer = tokenizer
        self.max_length = max_length
        # Получаем uuid из индекса
        self.uuids = list(df.index)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text = self.df.iloc[idx]["text"]
        uuid = self.uuids[idx]  # Берем из индекса

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "uuid": uuid
        }

In [ ]:
test_df = pd.read_parquet("test.parquet")

test_dataset = TestDataset(
    df=test_df,
    tokenizer=tokenizer,
    max_length=512
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False
)


model.eval()

submission = {
    "uuid": [],
    "integrity": [],
    "factuality": [],
    "truthfulness": []
}

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Inference"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        uuids = batch["uuid"]
        outputs = model(input_ids, attention_mask)
        integrity_preds = outputs["integrity"].argmax(dim=1).cpu().numpy()
        factuality_preds = outputs["factuality"].argmax(dim=1).cpu().numpy()
        truthfulness_preds = outputs["truthfulness"].argmax(dim=1).cpu().numpy()
        submission["uuid"].extend(uuids)
        submission["integrity"].extend(integrity_preds.tolist())
        submission["factuality"].extend(factuality_preds.tolist())
        submission["truthfulness"].extend(truthfulness_preds.tolist())


Inference:   0%|          | 0/313 [00:00<?, ?it/s]

In [ ]:
submission_df = pd.DataFrame(submission)

submission_df = submission_df[
    ["uuid", "integrity", "truthfulness", "factuality"]
]

submission_df.to_csv("submission.csv", index=False)

print("submission.csv сохранён")
submission_df.head()

submission.csv сохранён


,uuid,integrity,truthfulness,factuality
0,61f42863-9b53-4e78-a952-714da49f0c7a,1,1,1
1,f07bb53f-5639-46fe-9741-9465d516b8d4,1,1,1
2,f1d3d72b-3432-4604-8146-15a7d94b1598,1,1,1
3,935d8a6c-6d75-4321-a66d-f3061d267df7,1,1,1
4,c7f7f912-59f6-4db3-abc3-510f9318a3ff,1,1,1
